# QELM sur la matrice de mesures M, balayage en p

Expérience `experiments/QELM_M/exp_QELM_M.py` (issue #94). Pour chaque échantillon, le réservoir (TFIM désordonné, ancilla intriquée avec la charge du système) évolue pendant `evolution.t`, découpé en `Ns` pas. Après chaque pas, chaque qubit du système est mesuré en Z avec la probabilité `p`. Les features sont le relevé des mesures `M` (0 pour un qubit non mesuré), et la cible est `<Z_0>` de l'ancilla, jamais mesurée, à l'instant final.

Ce notebook lance le balayage avec `experiments/QELM_M/exp_QELM_M.sh`, relit les dossiers `outputs/exp_QELM_M/<date_heure>/N<N>_seed<seed>_p<p>/`, puis trace en fonction de `p` R2, NMSE, l'accuracy du signe de `<Z_0>` et les entropies (en bits) : celle de l'ancilla avec le système (von Neumann `S_vN` et Rényi d'ordre 1 `S_renyi_1`, issue #119), celle du secteur système sans l'ancilla (von Neumann `S_vN_sys` et Rényi d'ordre 2 `S_renyi_2_sys`) et la moyenne sur les qubits du système de l'entropie de chaque qubit seul (`S_vN_1q`, `S_renyi_2_1q`), issue #124. Chaque métrique est moyennée sur les seeds : à gauche la moyenne avec l'erreur standard en barres d'erreur, à droite l'erreur standard seule.

À p = 1, mesurer tout le système fixe la charge, donc `<Z_0>` vaut ±1 et se lit sur la dernière ligne de M : R2 ≈ 1 y est attendu quelle que soit la dynamique (issue #104).

L'entropie de l'ancilla est le paramètre d'ordre de purification de la transition induite par la mesure : `S = 1` bit tant que l'ancilla reste intriquée avec le système, `S = 0` quand les mesures l'ont purifiée. Rényi d'ordre 1 est la limite `n -> 1` de la famille de Rényi, c'est-à-dire l'entropie de von Neumann : les deux courbes coïncident.

L'état global reste pur (évolution unitaire et mesures projectives), donc l'entropie du secteur système est égale à celle de l'ancilla, à tout ordre de Rényi : la courbe `S_vN_sys` reproduit `S_vN`, et `S_renyi_2_sys` donne aussi l'entropie de Rényi-2 de l'ancilla (`S_2 <= S_vN`). Les moyennes mono-qubit `S_vN_1q` et `S_renyi_2_1q` sont, elles, une information nouvelle : l'intrication de chaque qubit du système avec tout le reste.

L'accuracy classe chaque échantillon de test selon le signe de `<Z_0>` (ancilla plus proche de `|0>` ou de `|1>`) et compte la part des signes que la prédiction retrouve.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so experiments/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Paramètres

Chaque valeur peut être changée ici, ou par la variable d'environnement du même nom (c'est ce que fait le test `tests/test_qelm_m_notebook.py` pour un petit balayage). `RUN = False` relit le dernier balayage sans en relancer un.

In [ ]:
L = os.environ.get("L", "500")  # samples per run
N_VALUES = os.environ.get("N_VALUES", "4")  # total qubits, ancilla included
SEEDS = os.environ.get("SEEDS", "42 43 44")
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.5 0.8 1.0")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Lancer le balayage

In [ ]:
from QELM.utils import latest_sweep, run_with_progress

if RUN:
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "OUT_ROOT": str(OUT_ROOT),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M.sh"], total=n_runs, env=env
    )

sweep_dir = latest_sweep(OUT_ROOT / "exp_QELM_M")
print(f"sweep: {sweep_dir}")

## Relire les métriques

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label, log scale). A metric missing from an older
# sweep is left out: the ancilla entropies came with issue #119, the
# accuracy and the system entropies with issue #124.
METRICS = [
    ("r2", "R2 (test)", False),
    ("nmse", "NMSE (test)", True),
    ("accuracy", "accuracy of sign <Z_0> (test)", False),
    ("S_vN_mean", "S_vN ancilla (bits)", False),
    ("S_renyi_1_mean", "S_Rényi-1 ancilla (bits)", False),
    ("S_vN_sys_mean", "S_vN system (bits)", False),
    ("S_renyi_2_sys_mean", "S_Rényi-2 system (bits)", False),
    ("S_vN_1q_mean", "mean single-qubit S_vN (bits)", False),
    ("S_renyi_2_1q_mean", "mean single-qubit S_Rényi-2 (bits)", False),
]
rows = load_sweep(sweep_dir)
METRICS = [m for m in METRICS if all(m[0] in r for r in rows)]
table = summarize(rows, by=("N", "p"), metrics=[key for key, _, _ in METRICS])
print(f"{len(rows)} runs")
print(
    f"{'N':>3} {'p':>5} {'runs':>4} " + " ".join(f"{key:>20}" for key, _, _ in METRICS)
)
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.4f} ± {r[f'{key}_sem']:<8.4f}" for key, _, _ in METRICS
    )
    print(f"{r['N']:>3} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

## Métriques en fonction de p

Une ligne par métrique : à gauche la moyenne sur les seeds ± l'erreur standard (`std / sqrt(n_runs)`), à droite l'erreur standard seule. Avec une seule seed l'erreur n'est pas définie et le graphe de droite reste vide.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    len(METRICS),
    2,
    figsize=(10, 3.5 * len(METRICS)),
    squeeze=False,
    constrained_layout=True,
)
for N in sorted({r["N"] for r in table}):
    sub = [r for r in table if r["N"] == N]
    p = [r["p"] for r in sub]
    for (key, label, log), (ax_mean, ax_err) in zip(METRICS, axes, strict=True):
        mean = np.array([r[f"{key}_mean"] for r in sub])
        sem = np.array([r[f"{key}_sem"] for r in sub])
        # One seed gives no spread: draw the means without error bars.
        err = np.nan_to_num(sem)
        # On a log axis a bar below 0 cannot be drawn and stretches the axis
        # down to ~0: stop the lower bar at half the mean.
        lower = np.minimum(err, 0.5 * mean) if log else err
        ax_mean.errorbar(
            p,
            mean,
            yerr=[lower, err],
            marker="o",
            capsize=3,
            label=f"N = {N}",
        )
        ax_err.plot(p, sem, marker="o", label=f"N = {N}")
for (key, label, log), (ax_mean, ax_err) in zip(METRICS, axes, strict=True):
    ax_mean.set(xlabel="measurement probability p", ylabel=label, title=f"{label} vs p")
    ax_err.set(
        xlabel="measurement probability p",
        ylabel=f"standard error of {key}",
        title=f"Error on {label} vs p",
    )
    if log:
        ax_mean.set_yscale("log")
    for ax in (ax_mean, ax_err):
        ax.grid(alpha=0.3)
        ax.legend()
figure_path = sweep_dir / "metrics_vs_p.png"
fig.savefig(figure_path, dpi=150)
print(f"figure: {figure_path}")
plt.show()